%pwd

In [2]:
%load_ext jupyter_black

In [4]:
from pathlib import Path

import pandas as pd
from joblib import Memory
from rapidfuzz import process
from tqdm import tqdm

cachedir = ".cache"
ROOT = Path("..")
memory = Memory(cachedir, verbose=0)

In [5]:
epc = pd.read_csv("address_profiling_testing_V6.csv", low_memory=False)
os = pd.read_csv("os_places_api.csv", low_memory=False)

In [6]:
epc.head()

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorConstruction,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area
0,1.000608e+11,NaN,NaN,"30, Pitt Street",PO33 3EB,43,NaN,NaN,NaN,7.8,...,Suspended,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,157,88,126.08
1,1.000608e+11,NaN,NaN,24 Upper Highland Road,PO33 1DZ,10,NaN,NaN,NaN,9.5,...,Suspended,NoInsulation,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,456,80,68.00
2,1.000608e+11,NaN,NaN,"66, Slade Road",PO33 1EG,63,NaN,NaN,NaN,3.0,...,Solid,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,202,57,82.00
3,1.000608e+11,NaN,NaN,"67, Mountfield Road, Wroxall",PO38 3BX,60,NaN,NaN,NaN,3.4,...,Solid,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,100,46,76.00
4,1.000624e+11,NaN,NaN,"Flat 40, Homebray House, Mary Rose Avenue",PO33 4LW,69,NaN,NaN,NaN,1.9,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,319,46,49.00


In [7]:
os.head()

,rpc,postcode,logical_status_code,country_code_description,x_coordinate,last_update_date,classification_code_description,organisation_name,udprn,status,...,y_coordinate,match_description,classification_code,entry_date,MATCH,country_code,_airbyte_ab_id,_airbyte_emitted_at,_airbyte_normalized_at,_airbyte_os_places_api_hashid
0,2,PO30 5AA,1,This record is within England,449564.00,08/07/2022,Self Contained Flat (Includes Maisonette / Apa...,NaN,28743180,APPROVED,...,89061.00,EXACT,RD06,10/10/2006,1,E,6c6b6cf8-37e1-461f-9560-84512fe8a237,2023-09-05 08:30:33.818+00,2023-09-05 08:37:23.597212+00,bdefa406d6ebfb346537bd28d6e87d9e
1,2,PO30 1BU,1,This record is within England,449512.00,22/06/2021,Semi-Detached,NaN,19452986,APPROVED,...,88857.00,EXACT,RD03,07/10/2003,1,E,e9f473a5-5f82-49ce-9b2f-98f0de26b5ad,2023-09-05 08:30:33.829+00,2023-09-05 08:37:23.597212+00,ee368cb83948badb3d8d6920785298a3
2,1,PO30 2JX,1,This record is within England,451537.00,29/06/2021,Semi-Detached,NaN,19458239,APPROVED,...,91399.00,EXACT,RD03,07/10/2003,1,E,43444046-b0f8-4fa9-a9ce-b2c79f1a4dd8,2023-09-05 08:30:33.83+00,2023-09-05 08:37:23.597212+00,334c9eb5430c5f0c23435234600f41a3
3,1,PO30 5LH,1,This record is within England,448049.98,01/05/2021,Semi-Detached,NaN,19463742,APPROVED,...,88874.58,EXACT,RD03,07/10/2003,1,E,5099b0d2-2237-4533-8a21-8b734e3dc3c9,2023-09-05 08:30:33.832+00,2023-09-05 08:37:23.597212+00,5d2af27a56c079e152bac36ac70e35a9
4,1,PO30 2HW,1,This record is within England,450698.00,08/07/2022,Terraced,NaN,19457943,APPROVED,...,89075.00,EXACT,RD04,07/10/2003,1,E,c1b60627-6ce8-497d-bb03-71eaa0326276,2023-09-05 08:30:33.833+00,2023-09-05 08:37:23.597212+00,f4fecfe0bc38f6898b80ac486b6de668


In [8]:
epc.iloc[0]

UPRN                                         100060765313.0
UDPRN                                                   NaN
ParityAddressId                                         NaN
Address                                     30, Pitt Street
PostCode                                           PO33 3EB
Environmental Impact Rating                              43
Environmental Impact Rating Band                        NaN
Fuel Bills (£/yr)                                       NaN
Realistic Fuel Bill (Regional)                          NaN
tCO2                                                    7.8
Heating Cost (£/yr)                                     NaN
Heating Cost (£/yr).1                                   NaN
SAP Rating                                               49
SAP Band                                                  E
Lodgement Date                                   2008-11-20
SAP version                                             NaN
PropertyType                            

In [9]:
epc[epc["UPRN"].isnull()][["UPRN", "Address", "PostCode"]]

,UPRN,Address,PostCode
10,NaN,"The Flat Above, Freshwater Post Office, Gate Lane",PO40 9PY
57,NaN,5A Mill Hill Road,PO31 7EA
60,NaN,"Unit 7, Walpan Farm, Southdown Lane, Chale",PO38 2JE
62,NaN,"Flat 1, St. Malo, 12 Osbourne Road",PO37 6BE
63,NaN,"Couthy Butt, Down Court Lane, Whitwell",PO38 2PJ
...,...,...,...
66154,NaN,Basement Flat 21-22 Cross Street,PO33 2AA
66168,NaN,"Whitfields P2, The West Bay Club",PO41 0RJ
66187,NaN,"Ground Floor Flat, Nodes Point Holiday Park, N...",PO33 1YA
66251,NaN,"Curlew Cottage, Salterns Village, Salterns Road",PO34 5AQ


In [34]:
epc["UPRN"].isna().sum()

1611

In [10]:
os[(os.postcode == "PO38 2PJ") & (os.address.str.contains(""))][
    ["postcode", "organisation_name", "address", "uprn", "parent_uprn", "udprn"]
]

,postcode,organisation_name,address,uprn,parent_uprn,udprn
72258,PO38 2PJ,NaN,"HERMITAGE COURT FARM, WHITWELL, VENTNOR, PO38 2PJ",10090466759,NaN,19512494
76074,PO38 2PJ,NaN,"DOWN COURT FARM, WHITWELL, VENTNOR, PO38 2PJ",10023713089,NaN,19512493


---

### fuzzymatcher doesn't work

!pip install fuzzymatcher

In [11]:
import fuzzymatcher

In [12]:
df_left = pd.DataFrame(
    {
        "address": [
            # "Flat 1, St. Malo, 12 Osbourne Road PO37 6BE",
            # "The Flat Above, Freshwater Post Office, Gate Lane PO40 9PY",
            # "5A Mill Hill Road PO31 7EA",
            # "Unit 7, Walpan Farm, Southdown Lane, Chale PO38 2JE",
            # "Couthy Butt, Down Court Lane, Whitwell PO38 2PJ",
            # "Basement Flat 21-22 Cross Street Ryde PO33 2AA",
            "Curlew Cottage, Salterns Village, Salterns Road PO34 5AQ",
        ],
    },
)
df_left

,address
0,"Curlew Cottage, Salterns Village, Salterns Roa..."


In [13]:
df_right = pd.DataFrame(
    {"address": os[os.postcode.str.contains("PO34")].address},
)  # .str.lower().tolist()
# choices = os[os.postcode == "PO37 6BE"].address.str.lower().tolist()

df_right

,address
44616,"OAKHILL MEWS COTTAGE, OAKHILL ROAD, SEAVIEW, P..."
44617,"3 FAIRY HILL, SEAVIEW LANE, SEAVIEW, PO34 5DG"
44618,"19, SEAVIEW BAY, PIER ROAD, SEAVIEW, PO34 5BP"
44619,"DOUBLOON, OAKHILL ROAD, SEAVIEW, PO34 5AP"
44620,"STONE HOUSE, PRIORY ROAD, SEAVIEW, PO34 5BU"
...,...
46341,"FLAT 4, SEAHILL HOUSE, PIER ROAD, SEAVIEW, PO3..."
46342,"13, EDDINGTON ROAD, SEAVIEW, PO34 5EE"
46343,"4, SANDPIPERS, OLD SEAVIEW LANE, SEAVIEW, PO34..."
46351,"19, ORCHARD ROAD, SEAVIEW, PO34 5JE"


In [14]:
%%time
df_both = fuzzymatcher.fuzzy_left_join(df_left, df_right, left_on="address", right_on="address")
df_both

CPU times: user 275 ms, sys: 5.36 ms, total: 280 ms
Wall time: 305 ms


,best_match_score,__id_left,__id_right,address_left,address_right
0,0.317478,0_left,159_right,"Curlew Cottage, Salterns Village, Salterns Roa...","RED CROSS COTTAGE, SALTERNS ROAD, SEAVIEW, PO3..."


In [15]:
df_both.address_right[0]

'RED CROSS COTTAGE, SALTERNS ROAD, SEAVIEW, PO34 5AG'

In [16]:
from pandarallel import pandarallel

In [17]:
pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 10 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


!pip install ipywidgets

In [18]:
epc_df = epc[epc["UPRN"].isna()].copy()
epc_df["address_postcode"] = epc_df.apply(lambda row: f"{row.Address} {row.PostCode}", axis=1)
epc_df["post_district"] = epc_df.PostCode.apply(lambda s: s.split(" ")[0])


# Function to get the closest match and its UPRN based on the address and postcode
@memory.cache
def match_address(address, district):
    df_left = pd.DataFrame({"address": [address]})

    df_right = pd.DataFrame({"address": os[os.postcode.str.contains(district)].address})

    df_both = fuzzymatcher.fuzzy_left_join(df_left, df_right, left_on="address", right_on="address")
    matched_address = df_both.address_right[0]

    return matched_address


epc_df["matched_address"] = epc_df.parallel_apply(
    lambda row: match_address(row["address_postcode"], row["post_district"]),
    axis=1,
)

In [ ]:
epc_df = epc_df.merge(
    os[["address", "uprn", "udprn"]],
    left_on="matched_address",
    right_on="address",
)
epc_df["UPRN"] = epc_df.uprn
epc_df["UDPRN"] = epc_df.udprn
epc_df = epc_df.drop(columns=["address_postcode", "post_district", "address", "uprn", "udprn"])

In [35]:
epc_df[epc_df.PostCode == "PO37 6BE"]

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area,matched_address
3,200001777505,50494010,NaN,"Flat 1, St. Malo, 12 Osbourne Road",PO37 6BE,48,NaN,NaN,NaN,5.1,...,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,103,107,103.00,"FLAT 1, ST. MALO, PALMERSTON ROAD, SHANKLIN, P..."
474,100062438440,19503513,NaN,"Flat 2, Rosemary Court, 8 Osborne Road",PO37 6BE,59,NaN,NaN,NaN,2.5,...,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,138,36,36.24,"8, OSBORNE ROAD, SHANKLIN, PO37 6BE"
475,100062438440,19503513,NaN,"Flat 3, Rosemary Court, 8 Osborne Road",PO37 6BE,48,NaN,NaN,NaN,3.3,...,NaN,NaN,NaN,0.0,0.0,Natural,139,37,36.81,"8, OSBORNE ROAD, SHANKLIN, PO37 6BE"
476,100062438440,19503513,NaN,"Flat 1, Rosemary Court, 8 Osborne Road",PO37 6BE,76,NaN,NaN,NaN,1.7,...,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,157,35,47.77,"8, OSBORNE ROAD, SHANKLIN, PO37 6BE"


In [43]:
epc[(epc.UPRN > 100006200000.0) & (epc.UPRN < 100006400000)]

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorConstruction,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area


In [25]:
epc_df.UPRN.value_counts()

100062442111    81
10067633995     63
10003339173     61
10003321460     41
10067634178     34
                ..
100062434667     1
10024250939      1
100062045239     1
10023713044      1
100062438081     1
Name: UPRN, Length: 835, dtype: int64

In [26]:
epc_df[epc_df.UPRN == 100062442111]

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area,matched_address
273,100062442111,57212008,NaN,"14 Rookley Country Park, Main Road, Rookley",PO38 3LU,80,NaN,NaN,NaN,1.4,...,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,72,35,52.09,"ROOKLEY COUNTRY PARK LTD, MAIN ROAD, VENTNOR, ..."
274,100062442111,57212008,NaN,"57 Rookley Country Park, Main Road, Rookley",PO38 3LU,78,NaN,NaN,NaN,2.1,...,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,89,51,53.22,"ROOKLEY COUNTRY PARK LTD, MAIN ROAD, VENTNOR, ..."
275,100062442111,57212008,NaN,"29 Rookley Country Park, Main Road, Rookley",PO38 3LU,80,NaN,NaN,NaN,1.4,...,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,72,35,52.09,"ROOKLEY COUNTRY PARK LTD, MAIN ROAD, VENTNOR, ..."
276,100062442111,57212008,NaN,"2 Rookley Country Park, Main Road, Rookley",PO38 3LU,76,NaN,NaN,NaN,1.6,...,LimitedInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,72,42,52.00,"ROOKLEY COUNTRY PARK LTD, MAIN ROAD, VENTNOR, ..."
277,100062442111,57212008,NaN,"39 Rookley Country Park, Main Road, Rookley",PO38 3LU,80,NaN,NaN,NaN,1.4,...,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,72,35,52.09,"ROOKLEY COUNTRY PARK LTD, MAIN ROAD, VENTNOR, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349,100062442111,57212008,NaN,"30 Rookley Country Park, Main Road, Rookley",PO38 3LU,80,NaN,NaN,NaN,1.4,...,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,72,35,52.09,"ROOKLEY COUNTRY PARK LTD, MAIN ROAD, VENTNOR, ..."
350,100062442111,57212008,NaN,"5 Rookley Country Park, Main Road, Rookley",PO38 3LU,76,NaN,NaN,NaN,1.6,...,LimitedInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,72,42,52.00,"ROOKLEY COUNTRY PARK LTD, MAIN ROAD, VENTNOR, ..."
351,100062442111,57212008,NaN,"53 Rookley Country Park, Main Road, Rookley",PO38 3LU,78,NaN,NaN,NaN,2.1,...,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,89,51,53.22,"ROOKLEY COUNTRY PARK LTD, MAIN ROAD, VENTNOR, ..."
352,100062442111,57212008,NaN,"51 Rookley Country Park, Main Road, Rookley",PO38 3LU,78,NaN,NaN,NaN,2.1,...,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,89,51,53.22,"ROOKLEY COUNTRY PARK LTD, MAIN ROAD, VENTNOR, ..."


In [24]:
epc_df[epc_df.UPRN.duplicated()]

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area,matched_address
8,10023713442,51392405,NaN,"Flat 2, 3 Bowling Green Lane",PO30 1RR,71,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,0.0,NaN,104,27,48.10,"3, BOWLING GREEN LANE, NEWPORT, PO30 1RR"
9,10023713442,51392405,NaN,"Flat 1, 3 Bowling Green Lane",PO30 1RR,62,NaN,NaN,NaN,2.4,...,NaN,NaN,NaN,0.0,0.0,Natural,156,70,53.00,"3, BOWLING GREEN LANE, NEWPORT, PO30 1RR"
11,10024249679,51653139,NaN,"Flat 8, Harbour View, Fairlee Road",PO30 2EA,84,NaN,NaN,NaN,1.2,...,NaN,NaN,NaN,NaN,0.0,NaN,71,42,55.07,"FLAT 10, HARBOUR QUAY, FAIRLEE ROAD, NEWPORT, ..."
12,10024249679,51653139,NaN,"Flat 17, Harbour View, Fairlee Road",PO30 2EA,82,NaN,NaN,NaN,1.5,...,NaN,NaN,NaN,NaN,0.0,NaN,84,49,66.71,"FLAT 10, HARBOUR QUAY, FAIRLEE ROAD, NEWPORT, ..."
13,10024249679,51653139,NaN,"Flat 20, Harbour View, Fairlee Road",PO30 2EA,85,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,0.0,NaN,69,32,45.99,"FLAT 10, HARBOUR QUAY, FAIRLEE ROAD, NEWPORT, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1589,10003319424,19503725,NaN,"Fir, Dove Court, 2 Crescent Road",PO37 6DG,49,NaN,NaN,NaN,3.1,...,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,308,56,46.00,"DOVE COURT HOLIDAY APARTMENTS, 2, CRESCENT ROA..."
1590,10003319424,19503725,NaN,"Elm, Dove Court, 2 Crescent Road",PO37 6DG,56,NaN,NaN,NaN,3.1,...,NaN,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,353,78,64.00,"DOVE COURT HOLIDAY APARTMENTS, 2, CRESCENT ROA..."
1597,100062044319,19484768,NaN,"Flat 8, 31 Weeks Road",PO33 2TL,85,NaN,NaN,NaN,0.9,...,NaN,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,129,77,43.00,"FLAT 1, 31, WEEKS ROAD, RYDE, PO33 2TL"
1598,100062044319,19484768,NaN,"Flat 7, 31 Weeks Road",PO33 2TL,85,NaN,NaN,NaN,0.8,...,NaN,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,121,67,36.00,"FLAT 1, 31, WEEKS ROAD, RYDE, PO33 2TL"


In [23]:
epc_df

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area,matched_address
0,10091031278,54585404,NaN,"The Flat Above, Freshwater Post Office, Gate Lane",PO40 9PY,56,NaN,NaN,NaN,4.2,...,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,94,62,94.0,"FLAT 1, THE PIANO CAFE, GATE LANE, FRESHWATER,..."
1,10091028614,53821399,NaN,5A Mill Hill Road,PO31 7EA,72,NaN,NaN,NaN,1.9,...,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,89,47,59.0,"10B, MILL HILL ROAD, COWES, PO31 7EA"
2,10095855821,56614164,NaN,"Unit 7, Walpan Farm, Southdown Lane, Chale",PO38 2JE,73,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,0.0,0.0,NaN,95,54,60.0,"7 WALPAN FARM, SOUTHDOWN LANE, CHALE, VENTNOR,..."
3,200001777505,50494010,NaN,"Flat 1, St. Malo, 12 Osbourne Road",PO37 6BE,48,NaN,NaN,NaN,5.1,...,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,103,107,103.0,"FLAT 1, ST. MALO, PALMERSTON ROAD, SHANKLIN, P..."
4,10090466759,19512494,NaN,"Couthy Butt, Down Court Lane, Whitwell",PO38 2PJ,66,NaN,NaN,NaN,3.2,...,NoInsulation,NaN,DoubleGlazingBefore2002,1.0,0.0,Natural,199,90,89.0,"HERMITAGE COURT FARM, WHITWELL, VENTNOR, PO38 2PJ"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1606,10003321617,19494716,NaN,"Aida Cottage, 30 Forelands Field Road",PO35 5TR,49,NaN,NaN,NaN,3.8,...,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,233,116,71.0,"29-30, FORELANDS FIELD ROAD, BEMBRIDGE, PO35 5TR"
1607,10003321617,19494716,NaN,"Rose Cottage, 29 Forelands Field Road",PO35 5TR,40,NaN,NaN,NaN,5.7,...,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,568,198,93.0,"29-30, FORELANDS FIELD ROAD, BEMBRIDGE, PO35 5TR"
1608,100062447444,19518389,NaN,"Glassworks, Madeira Lane, Isle Of Wight",PO40 9SP,81,NaN,NaN,NaN,4.2,...,NaN,NaN,NaN,0.0,0.0,NaN,270,271,292.0,"MADEIRA LODGE, MADEIRA LANE, FRESHWATER, PO40 9SP"
1609,10091028385,53009224,NaN,Basement Flat 21-22 Cross Street,PO33 2AA,58,NaN,NaN,NaN,5.1,...,NoInsulation,NaN,DoubleGlazing,0.0,0.0,Natural,105,152,146.0,"BASEMENT FLAT, 22, CROSS STREET, RYDE, PO33 2AA"


In [67]:
from rapidfuzz import fuzz, process

In [83]:
# choices = os.address.str.lower().tolist()
choices = os[os.postcode == "PO37 6BE"].address.str.lower().tolist()

In [84]:
malo = "Flat 1, St. Malo, 12 Osbourne Road PO37 6BE".lower()
malo

'flat 1, st. malo, 12 osbourne road po37 6be'

In [85]:
process.extract(malo, choices, scorer=fuzz.WRatio, limit=2)

[('flat 1, 6, osborne road, shanklin, po37 6be', 72.90697674418604, 10),
 ('flat 4, 10, osborne road, shanklin, po37 6be', 72.06896551724137, 15)]

In [86]:
process.extractOne(malo, choices, scorer=fuzz.WRatio)

('flat 1, 6, osborne road, shanklin, po37 6be', 72.90697674418604, 10)

In [87]:
from rapidfuzz import fuzz, process, utils

In [88]:
process.extract(malo, choices, scorer=fuzz.WRatio, limit=2, processor=utils.default_process)

[('flat 1, 6, osborne road, shanklin, po37 6be', 74.55696202531645, 10),
 ('flat 2, 6, osborne road, shanklin, po37 6be', 74.55696202531645, 13)]

In [ ]:
import pandas as pd
from rapidfuzz import process
from tqdm import tqdm

# Sample data (Replace these with your actual DataFrames)
epc_data = epc[epc["UPRN"].isna()].copy()
os_data = os.copy()

epc_df = pd.DataFrame(epc_data)
os_df = pd.DataFrame(os_data)


# Function to get the closest match and its UPRN based on the address and postcode
def match_address(address, postcode, choices_df):
    combined_string = address + " " + postcode
    # Using the `extractOne` method to get the best match
    best_match = process.extractOne(
        combined_string,
        choices_df["address"] + " " + choices_df["postcode"],
    )

    # Optional: Set a threshold for a match, e.g. 80
    if best_match[1] > 80:
        matched_uprn = choices_df.loc[
            choices_df["address"] + " " + choices_df["postcode"] == best_match[0],
            "uprn",
        ].iloc[0]
        return matched_uprn
    else:
        return None


# Using tqdm in the apply function for a progress bar
tqdm.pandas()
epc_df["UPRN"] = epc_df.progress_apply(
    lambda row: match_address(row["Address"], row["PostCode"], os_df),
    axis=1,
)

In [64]:
epc_df

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorConstruction,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area
10,100062624793,NaN,NaN,"The Flat Above, Freshwater Post Office, Gate Lane",PO40 9PY,56,NaN,NaN,NaN,4.2,...,OtherPremisesBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,94,62,94.0
57,10003316026,NaN,NaN,5A Mill Hill Road,PO31 7EA,72,NaN,NaN,NaN,1.9,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,89,47,59.0
60,10024249819,NaN,NaN,"Unit 7, Walpan Farm, Southdown Lane, Chale",PO38 2JE,73,NaN,NaN,NaN,2.0,...,Other,NaN,NaN,NaN,0.0,0.0,NaN,95,54,60.0
62,10013855348,NaN,NaN,"Flat 1, St. Malo, 12 Osbourne Road",PO37 6BE,48,NaN,NaN,NaN,5.1,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,103,107,103.0
63,100062442401,NaN,NaN,"Couthy Butt, Down Court Lane, Whitwell",PO38 2PJ,66,NaN,NaN,NaN,3.2,...,Solid,NoInsulation,NaN,DoubleGlazingBefore2002,1.0,0.0,Natural,199,90,89.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66154,10024250503,NaN,NaN,Basement Flat 21-22 Cross Street,PO33 2AA,58,NaN,NaN,NaN,5.1,...,Solid,NoInsulation,NaN,DoubleGlazing,0.0,0.0,Natural,105,152,146.0
66168,10003315443,NaN,NaN,"Whitfields P2, The West Bay Club",PO41 0RJ,76,NaN,NaN,NaN,2.2,...,Suspended,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,215,191,85.0
66187,10013855098,NaN,NaN,"Ground Floor Flat, Nodes Point Holiday Park, N...",PO33 1YA,37,NaN,NaN,NaN,3.6,...,Solid,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,536,65,36.0
66251,nan,NaN,NaN,"Curlew Cottage, Salterns Village, Salterns Road",PO34 5AQ,47,NaN,NaN,NaN,5.1,...,Solid,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,825,156,104.0


In [63]:
epc_df["UPRN"] = epc_df["UPRN"].astype(str).apply(lambda s: s.replace(".0", ""))

In [66]:
epc_df[epc_df.PostCode == "PO37 6BE"]

,UPRN,UDPRN,ParityAddressId,Address,PostCode,Environmental Impact Rating,Environmental Impact Rating Band,Fuel Bills (£/yr),Realistic Fuel Bill (Regional),tCO2,...,FloorConstruction,FloorInsulation,FloorInsulationThickness,MultipleGlazingType,OpenFireplacesCount,Renewables,Ventilation,Water Heating Cost (£/yr),Lighting Cost (£/yr),Total Floor Area
62,10013855348,NaN,NaN,"Flat 1, St. Malo, 12 Osbourne Road",PO37 6BE,48,NaN,NaN,NaN,5.1,...,AnotherDwellingBelow,NaN,NaN,DoubleGlazingBefore2002,0.0,0.0,Natural,103,107,103.00
4738,10013854946,NaN,NaN,"Flat 2, Rosemary Court, 8 Osborne Road",PO37 6BE,59,NaN,NaN,NaN,2.5,...,Suspended,NoInsulation,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,138,36,36.24
29955,10013855712,NaN,NaN,"Flat 3, Rosemary Court, 8 Osborne Road",PO37 6BE,48,NaN,NaN,NaN,3.3,...,OtherPremisesBelow,NaN,NaN,NaN,0.0,0.0,Natural,139,37,36.81
50678,10013855348,NaN,NaN,"Flat 1, Rosemary Court, 8 Osborne Road",PO37 6BE,76,NaN,NaN,NaN,1.7,...,Suspended,Insulated,NaN,DoubleGlazingAfter2002,0.0,0.0,Natural,157,35,47.77


---

In [6]:
import pandas as pd
from rapidfuzz import fuzz, process
from tqdm import tqdm

# Sample data (Replace these with your actual DataFrames)
epc_data = epc[epc["UPRN"].isna()].copy()
os_data = os.copy()

epc_df = pd.DataFrame(epc_data)
os_df = pd.DataFrame(os_data)


# Function to get the closest match and its UPRN based on the address and postcode
def match_address(address, postcode, choices_df):
    combined_string = address + " " + postcode
    best_match = process.extractOne(
        combined_string,
        choices_df["address"] + " " + choices_df["postcode"],
        scorer=fuzz.token_sort_ratio,
    )

    if best_match[1] > 80:
        matched_uprn = choices_df.loc[
            choices_df["address"] + " " + choices_df["postcode"] == best_match[0],
            "uprn",
        ].iloc[0]
        return matched_uprn
    else:
        return None


# Using tqdm in the apply function for a progress bar
tqdm.pandas()
epc_df["UPRN"] = epc_df.progress_apply(
    lambda row: match_address(row["Address"], row["PostCode"], os_df),
    axis=1,
)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1611/1611 [01:44<00:00, 15.39it/s]


<IPython.core.display.Javascript object>

### Checking missing URPN address in both datasets (Hilliers)

In [43]:
epc["UPRN"] = epc.UPRN.astype(str)

<IPython.core.display.Javascript object>

In [44]:
epc.query("PostCode == 'PO38 3LT'")[["UPRN", "UDPRN", "PostCode", "Address"]].sort_values(
    by="Address",
)

,UPRN,UDPRN,PostCode,Address
117,nan,NaN,PO38 3LT,"The Hilliers, Pritchetts Way, Rookley"


<IPython.core.display.Javascript object>

In [45]:
os.query("postcode == 'PO38 3LT'")[["uprn", "udprn", "postcode", "address"]].sort_values(
    by="address",
)

,uprn,udprn,postcode,address
77012,100062442422,19514348,PO38 3LT,"A C VAN HIRE LTD, UNIT 5, PRITCHETTS WAY, ROOK..."
78416,100062442255,28581795,PO38 3LT,"BETAPAK, PRITCHETTS WAY, ROOKLEY, VENTNOR, PO3..."
76380,100062442253,19514338,PO38 3LT,"HILLIER MOTORS, PRITCHETTS WAY, ROOKLEY, VENTN..."
72193,10003335417,19514344,PO38 3LT,"HUNT FOREST GROUP, PRITCHETTS WAY, ROOKLEY, VE..."
74300,100062442262,19514339,PO38 3LT,"ISLAND MOTOR SALVAGE, PRITCHETTS WAY, ROOKLEY,..."
77993,10090467130,51204803,PO38 3LT,"JOHN PECK CONSTRUCTION LTD, UNIT 2-3, PRITCHET..."
72749,100062625396,19514336,PO38 3LT,"PRETTY WINDOWS, PRITCHETTS WAY, ROOKLEY, VENTN..."
74528,100062442423,19514349,PO38 3LT,"SKINNY MAMMOTH, UNIT 6, PRITCHETTS WAY, ROOKLE..."
78675,100062442457,19514347,PO38 3LT,"SWIFT JOINERY, UNIT 4, PRITCHETTS WAY, ROOKLEY..."
74278,100062442424,19514350,PO38 3LT,"THE WORXS, UNIT 7, PRITCHETTS WAY, ROOKLEY, VE..."


<IPython.core.display.Javascript object>